# TB Portals - **Agentic Rung 1**: RAD-DINO features + Balanced MSE

Phase 0 (cache frozen RAD-DINO embeddings once) + Rung 1 (light heads on cached features). Isolates the two highest-evidence floor-raisers **before** any agentic machinery: a CXR-foundation backbone + a balanced-regression loss + honest (val-Pearson, all-epoch) selection.

Mode **a2**: ALP head + cavity head -> Timika. Compared to BOTH our locked baseline (honest target) and Kantipudi (aspirational).

| | Romania | **Moldova** | Kazakhstan |
|---|---|---|---|
| locked A2 (beat this) | 20.11 | **30.68** | 21.35 |
| Kantipudi A2 (aspirational) | 18.70 | 18.85 | 19.62 |

**Attach dataset:** `tb-portals-cxr-pngs` only. **Internet: ON** (RAD-DINO downloads from Hugging Face; it is NOT gated). No MedSAM / crops / TBX needed.

## 0 - Clone  *(restart kernel after any pull that changed .py)*

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path: sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)
# After a git pull that changed .py modules, RESTART the kernel so Python reloads them.

## Install deps (transformers for RAD-DINO; torchxrayvision = fallback backbone)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torchxrayvision", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

## Paths

In [ ]:
import os
WORK          = "/kaggle/working"
REPO_DIR      = "/kaggle/working/dl-project-codebase"
DATASET       = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT = f"{DATASET}/kaggle_export"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
BACKBONE      = "rad-dino"                       # rad-dino (primary) | txrv (fallback) | densenet (control)
FEATURES      = f"{WORK}/features_{BACKBONE}.npz"
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("backbone:", BACKBONE, "-> features cache:", FEATURES)

## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

## 2 - Phase 0: cache frozen RAD-DINO features (~10-20 min, one-time)

In [ ]:
# Phase 0: cache FROZEN RAD-DINO features once (~10-20 min, downloads model from HF).
# Idempotent: skips if the cache already exists.
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
if os.path.isfile(FEATURES):
    print("features already cached ->", FEATURES)
else:
    from cache_features import main as cache_main
    cache_main(["--manifest", PAPER_MANIFEST, "--out", FEATURES,
                "--backbone", BACKBONE, "--batch-size", "32"])

## 3 - Rung 1a: backbone + plain MSE

In [ ]:
# Rung 1a: RAD-DINO features + plain-MSE head (isolates the BACKBONE lift).
from src.training.train_agentic import main as agentic_main
agentic_main(["--features", FEATURES, "--manifest", PAPER_MANIFEST,
              "--out-dir", f"{WORK}/checkpoints/agentic_a2_mse", "--mode", "a2",
              "--loss", "mse", "--select-metric", "pearson",
              "--held-outs", "Romania", "Moldova", "Kazakhstan", "--seeds", "0", "1", "2"])

## 4 - Rung 1b: + Balanced MSE (the Moldova-targeted loss)

In [ ]:
# Rung 1b: + Balanced MSE (isolates the LOSS fix for the sicker held-out country).
from src.training.train_agentic import main as agentic_main
agentic_main(["--features", FEATURES, "--manifest", PAPER_MANIFEST,
              "--out-dir", f"{WORK}/checkpoints/agentic_a2_bmc", "--mode", "a2",
              "--loss", "bmc", "--select-metric", "pearson",
              "--held-outs", "Romania", "Moldova", "Kazakhstan", "--seeds", "0", "1", "2"])

## 5 - Save (download -> baseline_runs/agentic/)

In [ ]:
import os, shutil
dst = f"{WORK}/agentic_rung1"
os.makedirs(dst, exist_ok=True)
for tag in ("mse", "bmc"):
    d = f"{WORK}/checkpoints/agentic_a2_{tag}"
    if os.path.isdir(d):
        shutil.copy(f"{d}/results_agentic.csv", f"{dst}/results_agentic_{tag}.csv")
shutil.copy(FEATURES, f"{dst}/{os.path.basename(FEATURES)}")   # reuse the cache next time
zip_path = shutil.make_archive(f"{WORK}/agentic_rung1", "zip", dst)
print("Saved ->", zip_path)
print("Download it; drop the two results CSVs into baseline_runs/agentic/ and keep the .npz to skip re-caching.")